[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Errors and Retries &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `outcome_of`,
`describe`, `worth_retrying`, `backoff` and `get_with_retries`. Run it first.


In [1]:
import importlib
import random
import sys
import time
import urllib.request
import uuid
from pathlib import Path

import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
RETRY_STATUSES = {408, 429, 500, 502, 503, 504}


def outcome_of(url, **kwargs):
    """The response to a GET, or the exception raised in its place."""
    try:
        return requests.get(url, **kwargs)
    except requests.exceptions.RequestException as error:
        return error


def describe(outcome):
    return outcome.status_code if isinstance(outcome, requests.Response) else type(outcome).__name__


def worth_retrying(outcome):
    """Whether a GET that ended in this outcome could succeed if it were sent again."""
    if isinstance(outcome, requests.Response):
        return outcome.status_code in RETRY_STATUSES
    return isinstance(outcome, (requests.exceptions.Timeout, requests.exceptions.ConnectionError))


def backoff(attempt, rng, cap=30):
    """Seconds to wait after failed attempt number attempt + 1: a random share of a longest that doubles."""
    return rng.uniform(0, min(cap, 2 ** attempt))


def get_with_retries(url, attempts=4, deadline=10, timeout=5, rng=None):
    """The response to a GET, sent again after each failure worth retrying, within the attempts and the deadline."""
    rng = rng or random.Random()
    headers = {"X-Request-Id": uuid.uuid4().hex}           # the same id on every attempt
    stop_at = time.monotonic() + deadline
    for attempt in range(1, attempts + 1):
        outcome = outcome_of(url, headers=headers, timeout=timeout)
        print(f"  attempt {attempt}: {describe(outcome)}")
        if not worth_retrying(outcome) or attempt == attempts:
            break
        wait = backoff(attempt - 1, rng)
        if time.monotonic() + wait > stop_at:
            print("  stopping: the next wait would pass the deadline")
            break
        time.sleep(wait)
    if isinstance(outcome, Exception):
        raise outcome
    return outcome


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A slow response inside its timeout.


In [2]:
start = time.monotonic()
response = requests.get(f"{BASE}/network/report", timeout=3)

print(response.status_code, "in", round(time.monotonic() - start), "seconds")


200 in 2 seconds


The report takes 2 seconds, inside the 3 allowed. A read timeout belongs above the slowest response a
program expects, and a deadline, not the timeout, caps how long the program waits in all.


**2.** The narrowest class that catches a closed connection.


In [3]:
try:
    requests.get(f"{BASE}/hang-up", timeout=5)
except requests.exceptions.ConnectionError as error:
    print(type(error).__name__, "| also a RequestException:", isinstance(error, requests.exceptions.RequestException))


ConnectionError | also a RequestException: True


`requests.exceptions.ConnectionError` is the narrowest class that catches it. `RequestException`
catches it too, along with every other failure requests raises.


**3.** Four decisions.


In [4]:
for label, url, timeout in [("504", f"{BASE}/status/504", 5), ("408", f"{BASE}/status/408", 5),
                            ("400", f"{BASE}/status/400", 5), ("trickle", f"{BASE}/trickle", 1)]:
    outcome = outcome_of(url, timeout=timeout)
    print(f"{label:<8} {describe(outcome)!s:<4} worth retrying: {worth_retrying(outcome)}")


504      504  worth retrying: True
408      408  worth retrying: True
400      400  worth retrying: False
trickle  200  worth retrying: False


A `504` and a `408` are worth another try, and a `400` is not: it needs a different request.
`/trickle` succeeded, in three seconds and inside a timeout of 1, and a success is never retried.


**4.** Attempts the server can count.


In [5]:
same = {"X-Request-Id": uuid.uuid4().hex}
for attempt in range(1, 4):
    print("attempt", attempt, describe(outcome_of(f"{BASE}/network/unstable", headers=same, timeout=5)))

print("a new id:", describe(outcome_of(f"{BASE}/network/unstable", headers={"X-Request-Id": uuid.uuid4().hex}, timeout=5)))


attempt 1 ConnectionError
attempt 2 503
attempt 3 200
a new id: ConnectionError


The server counted the three attempts because they shared an id, and took the request with a new id
for a first attempt. A retry that makes a new id every time looks to the server like a new request,
which for this endpoint means failing forever.


**5.** One retry, and a failure that needs two.


In [6]:
session = requests.Session()
session.mount("http://", HTTPAdapter(max_retries=Retry(total=1, status_forcelist=[503])))
try:
    response = session.get(f"{BASE}/network/unstable", headers={"X-Request-Id": uuid.uuid4().hex}, timeout=5)
    print(response.status_code)
except requests.exceptions.RequestException as error:
    print(type(error).__name__, "|", error)


RetryError | HTTPConnectionPool(host='127.0.0.1', port=8765): Max retries exceeded with url: /network/unstable (Caused by ResponseError('too many 503 error responses'))


`total=1` allows one retry in all. The closed connection used it, the `503` that answered the retry
had none left to use, and `Retry` raised `RetryError`. A third attempt would have succeeded, so `total`
is best chosen from how many failures in a row a service usually gives.


**6.** Attempts that run out on a response.


In [7]:
response = get_with_retries(f"{BASE}/status/500", attempts=3, rng=random.Random(5))
print(response.status_code)


  attempt 1: 500
  attempt 2: 500
  attempt 3: 500
500


All three attempts failed, and `get_with_retries` returned the last response instead of raising,
because a response, even one that failed, is something a program can decide about, as the **Status
Codes** notebook's `decide` did. Only an exception, which leaves no response behind, is raised again.


---

&#8592; **Back to:** [Errors and Retries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/12-errors-and-retries.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
